In [3]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.cluster import KMeans

In [ ]:
df = pd.read_csv("../data/scraped_last_10_dates.csv")
df = df.dropna(subset=['text'])

vectorizer = CountVectorizer(stop_words='english', max_features=50)

tf_matrix = vectorizer.fit_transform(df['text'])
feature_names = vectorizer.get_feature_names_out()
tf_sums = tf_matrix.sum(axis=0).A1

tf_df = pd.DataFrame({'Term': feature_names, 'Frequency': tf_sums})
tf_df = tf_df.sort_values(by='Frequency', ascending=False).reset_index(drop=True)

print("Top Term Frequencies:")
print(tf_df)

In [ ]:
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_features=50)

tfidf_matrix = tfidf_vectorizer.fit_transform(df['text'])
feature_names = tfidf_vectorizer.get_feature_names_out()
avg_tfidf_scores = tfidf_matrix.mean(axis=0).A1

tfidf_df = pd.DataFrame({'Term': feature_names, 'Average TF-IDF': avg_tfidf_scores})
tfidf_df = tfidf_df.sort_values(by='Average TF-IDF', ascending=False).reset_index(drop=True)

print("Top 50 Terms by Average TF-IDF Score:")
print(tfidf_df)

In [10]:
num_clusters = 7
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
kmeans.fit(tfidf_matrix)

feature_names = tfidf_vectorizer.get_feature_names_out()

order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]

print("Top terms per cluster:")
for i in range(num_clusters):
    top_words = [feature_names[ind] for ind in order_centroids[i, :10]]
    print(f"Cluster {i}: {', '.join(top_words)}")

Top terms per cluster:
Cluster 0: russian, ukrainian, forces, july, 20, oblast, strikes, reported, occupied, claimed
Cluster 1: russian, ukrainian, forces, july, 16, reported, oblast, 17, occupied, russia
Cluster 2: russian, forces, ukrainian, july, 22, oblast, reported, russia, military, footage
Cluster 3: russian, forces, ukrainian, ukraine, 14, july, russia, reported, air, oblast
Cluster 4: russian, ukrainian, forces, july, 15, reported, 14, russia, strikes, strike
Cluster 5: russian, ukrainian, forces, 21, july, 20, oblast, occupied, reported, kilometers
Cluster 6: russian, forces, ukrainian, july, 18, reported, oblast, struck, occupied, continued


In [12]:
vocab = [
    "drone", "drones", "uav", "uavs", "uas", "usv", "usvs", "ugv", "fpv", 
    "unpiloted", "unmanned", "uncrewed", "loitering munition", "kamikaze", 
    "bayraktar", "shahed", "geran", "lancet", "reconnaissance", "surveillance", 
    "drone strike", "electronic warfare", "ew", "jamming", "air defense", 
    "volunteer", "crowdfunding", "grassroots", "innovation", "3d printing", 
    "production", "scale", "capacity", "defense industry", "aerorozvidka", 
    "united24", "general staff", "crimea", "black sea", "frontline"
]

vocab = list(set([v.lower() for v in vocab]))

tv = TfidfVectorizer(vocabulary=vocab, ngram_range=(1, 2))
tfidf_matrix = tv.fit_transform(df['text'])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans.fit(tfidf_matrix)

df['Cluster'] = kmeans.labels_

feature_names = tfidf_vectorizer.get_feature_names_out()

order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]

print("Top terms per cluster:")
for i in range(3):
    top_words = [feature_names[ind] for ind in order_centroids[i, :10]]
    print(f"Cluster {i}: {', '.join(top_words)}")

Top terms per cluster:
Cluster 0: drones, military, occupied, published, 21, 18, did, oblast, infrastructure, footage
Cluster 1: occupied, published, military, drones, oblast, did, 21, 18, range, footage
Cluster 2: isw, military, occupied, oblast, footage, published, drones, infrastructure, continued, range
